In [1]:
#creating consumer, pulls data
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'green-trips'

In [7]:
#deserializer
import json
import sys, os
sys.path.append(os.path.abspath("../src"))
from models import Ride, ride_deserializer

In [9]:
#
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-console',
    #value_deserializer=lambda x: json.loads(x)
    value_deserializer=ride_deserializer
)

In [10]:
#it should continue running until it receive a ride
record = next(consumer)

In [11]:
record

ConsumerRecord(topic='green-trips', partition=0, leader_epoch=1, offset=0, timestamp=1773460177668, timestamp_type=0, key=None, value=Ride(PULocationID=247, DOLocationID=69, trip_distance=0.7, total_amount=10.0, lpep_pickup_datetime='1759278107000', lpep_dropoff_datetime='1759278277000', passenger_count=1, tip_amount=1.7), headers=[], checksum=None, serialized_key_size=-1, serialized_value_size=209, serialized_header_size=-1)

In [12]:
record.value

Ride(PULocationID=247, DOLocationID=69, trip_distance=0.7, total_amount=10.0, lpep_pickup_datetime='1759278107000', lpep_dropoff_datetime='1759278277000', passenger_count=1, tip_amount=1.7)

In [14]:
#iterating over the records
from datetime import datetime

print(f"Listening to {topic_name}...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(int(ride.lpep_pickup_datetime) / 1000)
    print(f"Received: PU={ride.PULocationID}, DO={ride.DOLocationID}, "
          f"distance={ride.trip_distance}, amount=${ride.total_amount:.2f}, "
          f"pickup={pickup_dt}"
          f"dropoff={datetime.fromtimestamp(int(ride.lpep_dropoff_datetime) / 1000)}"
          f"passengers={ride.passenger_count}"
          f"tip_amount={ride.tip_amount}"
          )
    count += 1
    if count >= 10:
        print(f"\n... received {count} messages so far (stopping after 10 for demo)")
        break

consumer.close()

Listening to green-trips...
Received: PU=244, DO=244, distance=0.0, amount=$13.20, pickup=2025-10-01 00:16:44dropoff=2025-10-01 00:16:47passengers=1tip_amount=2.2
Received: PU=95, DO=170, distance=10.37, amount=$67.85, pickup=2025-10-01 00:07:36dropoff=2025-10-01 00:32:14passengers=1tip_amount=11.31
Received: PU=82, DO=138, distance=4.07, amount=$34.12, pickup=2025-09-30 21:10:29dropoff=2025-09-30 21:22:30passengers=1tip_amount=6.82
Received: PU=129, DO=37, distance=7.13, amount=$47.12, pickup=2025-09-30 21:49:46dropoff=2025-10-01 21:18:42passengers=1tip_amount=9.42
Received: PU=95, DO=134, distance=1.13, amount=$13.88, pickup=2025-10-01 00:11:24dropoff=2025-10-01 00:18:48passengers=1tip_amount=2.08
Received: PU=95, DO=70, distance=6.01, amount=$34.32, pickup=2025-10-01 00:07:08dropoff=2025-10-01 00:21:46passengers=1tip_amount=5.72
Received: PU=181, DO=137, distance=6.45, amount=$41.88, pickup=2025-10-01 00:36:08dropoff=2025-10-01 00:54:59passengers=1tip_amount=6.98
Received: PU=74, DO